# 05 — Project: Fashion-MNIST Classifier with Keras

## What this notebook covers

We now leave our from-scratch library and use **Keras** to solve a real image classification task:
the [Fashion-MNIST dataset](https://github.com/zalandoresearch/fashion-mnist) — 70,000 greyscale images
of 10 clothing categories.

This notebook shows how everything we built manually (layers, activations, loss, optimiser, backprop)
maps directly to the Keras API — which handles all the plumbing for us.

### Dataset classes
| Label | Item        |
|-------|-------------|
| 0     | T-shirt/top |
| 1     | Trouser     |
| 2     | Pullover    |
| 3     | Dress       |
| 4     | Coat        |
| 5     | Sandal      |
| 6     | Shirt       |
| 7     | Sneaker     |
| 8     | Bag         |
| 9     | Ankle boot  |


## 1. Load and preprocess the data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Load
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Split off a validation set and normalise to [0, 1]
X_valid, X_train = X_train_full[:5000] / 255., X_train_full[5000:] / 255.
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test           = X_test / 255.

print(f'Train: {X_train.shape}  |  Valid: {X_valid.shape}  |  Test: {X_test.shape}')


## 2. Visualise some samples

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='binary')
    ax.set_title(CLASS_NAMES[y_train[i]])
    ax.axis('off')
plt.suptitle('Fashion-MNIST samples', fontsize=13)
plt.tight_layout(); plt.show()


## 3. Build the model

Our from-scratch network had:
- `DenseLayer(2, 64)` + `ReLU` + `DenseLayer(64, 3)` + Softmax

In Keras the exact same architecture is:
```python
keras.layers.Dense(64, activation='relu')
keras.layers.Dense(3,  activation='softmax')
```
We just add a `Flatten` layer first to collapse the 28×28 images into a 784-d vector.


In [ ]:
keras.backend.clear_session()
np.random.seed(42); tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),   # 28*28 = 784 inputs
    keras.layers.Dense(300, activation='relu'),
    keras.layers.Dense(100, activation='relu'),
    keras.layers.Dense(10,  activation='softmax') # 10 classes
])

model.summary()


## 4. Compile and train

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',  # same loss we implemented by hand
    optimizer='sgd',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    validation_data=(X_valid, y_valid)
)


## 5. Plot learning curves

In [ ]:
import pandas as pd

df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[['loss', 'val_loss']].plot(ax=axes[0])
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)

df[['accuracy', 'val_accuracy']].plot(ax=axes[1])
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].grid(True)

plt.suptitle('Fashion-MNIST Training', fontsize=13)
plt.tight_layout(); plt.show()


## 6. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss:     {test_loss:.4f}')


## 7. Inspect predictions

In [ ]:
probs = model.predict(X_test[:10])
preds = np.argmax(probs, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i], cmap='binary')
    correct = preds[i] == y_test[i]
    color   = 'green' if correct else 'red'
    ax.set_title(f'Pred: {CLASS_NAMES[preds[i]]}\nTrue: {CLASS_NAMES[y_test[i]]}', color=color, fontsize=8)
    ax.axis('off')
plt.suptitle('Predictions (green = correct, red = wrong)', fontsize=12)
plt.tight_layout(); plt.show()


## 8. From scratch → Keras: the mapping

| What we built | Keras equivalent |
|---|---|
| `DenseLayer(n_inputs, n_neurons)` | `keras.layers.Dense(n_neurons)` |
| `ReLU` | `activation='relu'` |
| `SoftmaxWithCrossEntropy` | `activation='softmax'` + `loss='sparse_categorical_crossentropy'` |
| `Adam(lr=0.001)` | `optimizer='adam'` |
| `DropoutLayer(0.2)` | `keras.layers.Dropout(0.2)` |
| Training loop | `model.fit(...)` |

Everything is the same — Keras just removes the boilerplate so you can focus on architecture.
